# UdaPlay — Notebook 02: AI Research Agent

Demonstrates the full **UdaPlay agent workflow** using all **7 tools**:

```
User question
  → search_memory        (long-term JSON memory)
  → retrieve_game        (ChromaDB semantic search)
  → evaluate_retrieval   (LLM-as-judge)
  → [game_web_search]    (Tavily, if needed)
  → [save_memory]        (persist web results)
  → [summarize_game_profile]  (optional game profile)
  → format_report        (structured JSON output)
```

**Prerequisites:** Run `01_rag_pipeline.ipynb` first to populate ChromaDB.  
**API keys:** Set `OPENAI_API_KEY` and `TAVILY_API_KEY` in `.env`.


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent / "src"))

In [2]:
# SQLite shim — only needed on older Udacity workspace environments
import importlib.util
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules["sqlite3"] = pysqlite3
    print("pysqlite3 shim applied")
else:
    print("Standard sqlite3 — no shim needed")

Standard sqlite3 — no shim needed


In [3]:
import os, json, pathlib
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

from udaplay.config import Settings
from udaplay.agents import Agent
from udaplay.tools import make_game_tools
from udaplay.long_term_memory import LongTermMemory

load_dotenv()
settings = Settings()
print(f"Model : {settings.llm_model}")
print(f"OpenAI key loaded : {bool(settings.openai_api_key)}")
print(f"Tavily key loaded : {bool(settings.tavily_api_key)}")

Model : gpt-4o-mini
OpenAI key loaded : True
Tavily key loaded : True


## Connect to ChromaDB

In [4]:
CHROMA_PATH = pathlib.Path().resolve().parent / "chromadb"

chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=settings.openai_api_key,
    model_name=settings.embedding_model,
)

collection = chroma_client.get_or_create_collection(
    name=settings.collection_name,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

print(f"Collection '{collection.name}' — {collection.count()} documents")

Collection 'udaplay_games' — 15 documents


## Initialise Long-Term Memory

In [5]:
MEMORY_FILE = pathlib.Path().resolve().parent / "data" / "memory" / "long_term_memory.json"
MEMORY_FILE.parent.mkdir(parents=True, exist_ok=True)

memory = LongTermMemory(MEMORY_FILE)
print(f"Long-term memory: {memory.count()} entries stored at {MEMORY_FILE}")

Long-term memory: 0 entries stored at /Users/suleimanadebowaleojo/Claude/Projects/UdaPlay - An AI Research Agent for the Video Game Industry/data/memory/long_term_memory.json


## Build Tools + Agent

In [6]:
(retrieve_game,
 evaluate_retrieval,
 game_web_search,
 search_memory,
 save_memory,
 summarize_game_profile,
 format_report) = make_game_tools(collection, settings=settings, memory=memory)

print("7 tools created:", [t.name for t in [
    retrieve_game, evaluate_retrieval, game_web_search,
    search_memory, save_memory, summarize_game_profile, format_report,
]])

7 tools created: ['retrieve_game', 'evaluate_retrieval', 'game_web_search', 'search_memory', 'save_memory', 'summarize_game_profile', 'format_report']


In [7]:
INSTRUCTIONS = (
    "You are UdaPlay, an expert AI research agent specialising in the video game industry.\n\n"
    "Follow this workflow for EVERY question:\n"
    "1. Call `search_memory` to check long-term memory for previous research.\n"
    "2. Call `retrieve_game` to search the local vector database.\n"
    "3. Call `evaluate_retrieval` to judge the quality of local results.\n"
    "4. Decide the source:\n"
    "   - If `evaluate_retrieval` returns high/medium confidence: answer from local RAG.\n"
    "   - If `search_memory` had a strong match AND local RAG is weak: answer from memory.\n"
    "   - If both are insufficient: call `game_web_search`.\n"
    "5. Optionally call `summarize_game_profile` when the user asks for details about a specific game.\n"
    "6. Always call `format_report` as the final step with the complete answer.\n"
    "7. If the answer is useful (confidence medium or high), call `save_memory`.\n\n"
    "Always cite your sources. Be explicit about whether the answer came from "
    "Local RAG, Memory, or Web Search."
)

agent = Agent(
    model_name=settings.llm_model,
    instructions=INSTRUCTIONS,
    tools=[
        retrieve_game, evaluate_retrieval, game_web_search,
        search_memory, save_memory, summarize_game_profile, format_report,
    ],
    temperature=0.0,
)
print("Agent ready.")

Agent ready.


## Query Runner

In [8]:
def run_query(question: str, session_id: str = "notebook_session") -> dict:
    print(f"\n{'='*68}\nQUESTION: {question}\n{'='*68}")
    run = agent.invoke(question, session_id=session_id)
    state = run.get_final_state()

    # Print tool trace
    for msg in state.get("messages", []):
        role = getattr(msg, "role", "")
        if role == "tool":
            print(f"  [tool] {getattr(msg, 'name', '')} → {str(getattr(msg, 'content', ''))[:120]}")

    # Print final answer
    answer = state.get("answer") or state.get("content") or "(no final answer)"
    if isinstance(answer, list):
        answer = " ".join(str(a) for a in answer)
    print(f"\nANSWER:\n{str(answer)[:600]}")
    return state


## Example Queries

### Q1 — FIFA 21 developer *(not in local dataset → web fallback expected)*

In [9]:
r1 = run_query("Who developed FIFA 21?")


QUESTION: Who developed FIFA 21?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
  [tool] search_memory → "{\"found\": false, \"matches\": []}"
  [tool] retrieve_game → "Retrieved 3 result(s) from local database:\n\n[Result 1]\n  Title     : Halo Infinite\n  Platform  : Xbox Series X|S\n 
  [tool] evaluate_retrieval → "{\"confidence\": \"low\", \"useful\": false, \"description\": \"The retrieved documents do not contain any i

### Q2 — God of War Ragnarök release *(not in dataset → web fallback expected)*

In [10]:
r2 = run_query("When was God of War Ragnarök released?")


QUESTION: When was God of War Ragnarök released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
  [tool] search_memory → "{\"found\": false, \"matches\": []}"
  [tool] retrieve_game → "Retrieved 3 result(s) from local database:\n\n[Result 1]\n  Title     : Halo Infinite\n  Platform  : Xbox Series X|S\n 
  [tool] evaluate_retrieval → "{\"confidence\": \"low\", \"useful\": false, \"description\": \"The retrieved documents do n

### Q3 — Pokémon Gold and Silver *(in local dataset → local RAG expected)*

In [11]:
r3 = run_query("When was Pokémon Gold and Silver released?")


QUESTION: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
  [tool] search_memory → "{\"found\": false, \"matches\": []}"
  [tool] retrieve_game → "Retrieved 3 result(s) from local database:\n\n[Result 1]\n  Title     : Halo Infinite\n  Platform  : Xbox Series X|S\n 
  [tool] evaluate_retrieval → "{\"confidence\": \"low\", \"useful\": false, \"description\": \"The retrieved documents do not contain any information 
  [tool] game_web_search → "Web search results from Tavily

### Q4 — Memory reuse *(repeat Q1 in same session → memory expected)*

In [12]:
r4 = run_query("Who made FIFA 21?")


QUESTION: Who made FIFA 21?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
  [tool] search_memory → "{\"found\": false, \"matches\": []}"
  [tool] retrieve_game → "Retrieved 3 result(s) from local database:\n\n[Result 1]\n  Title     : Halo Infinite\n  Platform  : Xbox Series X|S\n 
  [tool] evaluate_retrieval → "{\"confidence\": \"low\", \"useful\": false, \"description\": \"The retrieved documents do not contain any inform

### Q5 — Game profile summary *(triggers summarize_game_profile)*

In [13]:
r5 = run_query("Give me a detailed profile of Pokémon Gold and Silver.")


QUESTION: Give me a detailed profile of Pokémon Gold and Silver.
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
  [tool] search_memory → "{\"found\": false, \"matches\": []}"
  [tool] retrieve_game → "Retrieved 3 result(s) from local database:\n\n[Result 1]\n  Title     : Halo Infinite\n  Platform  : Xbox Series X|S\n 
  [tool] evaluate_retrieval → "{\"confidence\": \"low\", \"useful\": false, \"description\": \"The retrieved documents do not contain any information 
  [tool] game_web_search → "Web search results from Tavily:\n\nDirect answer: FIFA 21 was developed by EA Canada and published by Electronic Arts. 
  [tool] format_report → "{\n  \"question\": \"Who developed FIFA 21?\",\n  \"

## Memory Inspection

In [14]:
entries = memory.all_entries()
print(f"Total long-term memory entries: {len(entries)}\n")
for i, e in enumerate(entries[-5:], 1):
    print(f"[{i}] Q: {e.get('question', '')}")
    print(f"    A: {str(e.get('answer', ''))[:120]}")
    print(f"    Source: {e.get('source_type', '')} | Confidence: {e.get('confidence', '')} | Tags: {e.get('tags', [])}")
    print()

Total long-term memory entries: 0

